# 🤗 x 🦾: Training SmolVLA with LeRobot Notebook

Welcome to the **LeRobot SmolVLA training notebook**! This notebook provides a ready-to-run setup for training imitation learning policies using the [🤗 LeRobot](https://github.com/huggingface/lerobot) library.

In this example, we train an `SmolVLA` policy using a dataset hosted on the [Hugging Face Hub](https://huggingface.co/), and optionally track training metrics with [Weights & Biases (wandb)](https://wandb.ai/).

## ⚙️ Requirements
- A Hugging Face dataset repo ID containing your training data (`--dataset.repo_id=YOUR_USERNAME/YOUR_DATASET`)
- Optional: A [wandb](https://wandb.ai/) account if you want to enable training visualization
- Recommended: GPU runtime (e.g., NVIDIA A100) for faster training

## ⏱️ Expected Training Time
Training with the `SmolVLA` policy for 20,000 steps typically takes **about 5 hours on an NVIDIA A100** GPU. On less powerful GPUs or CPUs, training may take significantly longer!

## Example Output
Model checkpoints, logs, and training plots will be saved to the specified `--output_dir`. If `wandb` is enabled, progress will also be visualized in your wandb project dashboard.


## Install LeRobot
This cell clones the `lerobot` repository from Hugging Face, installs FFmpeg, and installs the package in editable mode with train, dataset and smolvla features.

In [ ]:
#!git clone https://github.com/huggingface/lerobot.git
!git clone https://github.com/rshorton/lerobot.git
!apt-get install ffmpeg
!cd lerobot && pip install -e ".[train, dataset, smolvla]"

Cloning into 'lerobot'...
remote: Enumerating objects: 15538, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 15538 (delta 7), reused 7 (delta 7), pack-reused 15523 (from 2)
Receiving objects: 100% (15538/15538), 112.50 MiB | 14.21 MiB/s, done.
Resolving deltas: 100% (7974/7974), done.
Filtering content: 100% (45/45), 69.03 MiB | 4.82 MiB/s, done.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
Obtaining file:///content/lerobot
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build whe

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Weights & Biases login (optional)
This cell logs you into Weights & Biases (wandb) to enable experiment tracking and logging. This step is optional, you can skip it. If you want to use W&B remember to change `--wandb.enable` to true in the next section.

In [ ]:
!wandb login

For training I copied my training data to the instance instead of using Huggingface to store the data.  Also, I saved the checkpoints to my Google Drive.  The --dataset.root option below points to the local data.

In [ ]:
!tar -xzvf /content/drive/MyDrive/063024_test1.tar.gz

063024/
063024/test1/
063024/test1/data/
063024/test1/data/chunk-000/
063024/test1/data/chunk-000/file-000.parquet
063024/test1/videos/
063024/test1/videos/observation.images.side/
063024/test1/videos/observation.images.side/chunk-000/
063024/test1/videos/observation.images.side/chunk-000/file-000.mp4
063024/test1/videos/observation.images.wrist/
063024/test1/videos/observation.images.wrist/chunk-000/
063024/test1/videos/observation.images.wrist/chunk-000/file-000.mp4
063024/test1/videos/observation.images.front/
063024/test1/videos/observation.images.front/chunk-000/
063024/test1/videos/observation.images.front/chunk-000/file-000.mp4
063024/test1/meta/
063024/test1/meta/tasks.parquet
063024/test1/meta/episodes/
063024/test1/meta/episodes/chunk-000/
063024/test1/meta/episodes/chunk-000/file-000.parquet
063024/test1/meta/stats.json
063024/test1/meta/info.json


In [ ]:
import os
snapshot_dir = "/content/drive/MyDrive/lerobot_snapshots"
os.makedirs(snapshot_dir, exist_ok=True)

In [ ]:
!lerobot-train \
  --dataset.root=/content/063024/test1/ \
  --dataset.repo_id='063024/test1'  \
  --output_dir=/content/drive/MyDrive/lerobot_snapshots/smolvla_so101_063026_6dof_6 \
  --job_name=smolvla_so101_063026_6dof \
  --policy.path=lerobot/smolvla_base \
  --batch_size=64 \
  --steps=20000 \
  --save_freq=2000 \
  --policy.repo_id='063024/test1' \
  --policy.push_to_hub=false\
  --policy.device=cuda \
  --wandb.enable=true \
  --rename_map='{"observation.images.left": "observation.images.side", "observation.images.top": "observation.images.wrist", "observation.images.right": "observation.images.front"}' \
  --policy.empty_cameras=1 \
  --policy.input_features='{"observation.images.side": {"type":"VISUAL","shape":[3,480,640]}, "observation.images.wrist": {"type":"VISUAL","shape":[3,480,640]},"observation.images.front": {"type":"VISUAL","shape":[3,480,640]}}' \
  --policy.train_expert_only=true \
  --policy.freeze_vision_encoder=true

2026-06-30 21:32:38.097465: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-30 21:32:38.166338: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
config.json: 2.30kB [00:00, 11.0MB/s]
INFO 2026-06-30 21:32:49 ot_train.py:195 {'batch_size': 64,
 'checkpoint_path': None,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': 

In [ ]:
from google.colab import drive
drive.flush_and_unmount()